# Multilingual Image Search with Jina CLIP v2 and Elasticsearch

In a [previous article](https://www.elastic.co/search-labs/blog/openai-clip-alternatives), we explored open-source alternatives to OpenAI's CLIP for multimodal search. In this notebook, we take it further with **Jina CLIP v2**, a multilingual, multimodal embedding model, and use **Elasticsearch** as the vector search engine.

We'll build a multilingual image search system where users can query in **any language** and get relevant image results.

## Setup

In [ ]:
%pip install -qU datasets requests pillow elasticsearch dotenv matplotlib

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
JINA_API_KEY = os.getenv("JINA_API_KEY")

## Connect to Elasticsearch and create the index

In [ ]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY)
es_client.info()

In [ ]:
INDEX_NAME = "clip-v2-stock-images"

if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)

es_client.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "image_embedding": {
                "type": "dense_vector",
                "dims": 1024,
                "index": True,
                "similarity": "cosine",
            },
            "tags": {
                "type": "text",
                "fields": {"keyword": {"type": "keyword"}},
            },
        }
    },
)

## Jina CLIP v1 vs v2

| Feature | [Jina CLIP v1](https://huggingface.co/jinaai/jina-clip-v1) | [Jina CLIP v2](https://huggingface.co/jinaai/jina-clip-v2) |
| :---- | :---- | :---- |
| **Languages** | English only | 89 languages |
| **Max image resolution** | 224x224 | 512x512 |
| **Text encoder** | JinaBERT | Jina XLM-RoBERTa |
| **Matryoshka Representations** | No | Yes |
| **Embedding dimensions** | 768 | 1024 |
| **Max text length** | 512 tokens | 8192 tokens |

## Jina Embeddings API

We use the [Jina Embeddings API](https://jina.ai/embeddings/), a lightweight REST API that handles both text and image embeddings.

In [ ]:
import requests
import base64
from io import BytesIO

JINA_API_URL = "https://api.jina.ai/v1/embeddings"
JINA_HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {JINA_API_KEY}",
}


def image_to_base64(image, max_size=512):
    """Convert a PIL image to a base64 data URL, resizing to max_size."""

    image = image.copy()
    image.thumbnail((max_size, max_size))
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    return f"data:image/png;base64,{b64}"


def encode_texts(texts, dimensions=1024):
    """Encode a list of text strings using Jina CLIP v2 API."""

    data = {
        "input": [{"text": t} for t in texts],
        "model": "jina-clip-v2",
        "dimensions": dimensions,
    }

    response = requests.post(JINA_API_URL, headers=JINA_HEADERS, json=data)
    response.raise_for_status()

    return [item["embedding"] for item in response.json()["data"]]


def encode_images(images, dimensions=1024):
    """Encode a list of PIL images using Jina CLIP v2 API."""

    data = {
        "input": [{"image": image_to_base64(img)} for img in images],
        "model": "jina-clip-v2",
        "dimensions": dimensions,
    }

    response = requests.post(JINA_API_URL, headers=JINA_HEADERS, json=data)

    if not response.ok:
        print(f"Error {response.status_code}: {response.text}")

    response.raise_for_status()

    return [item["embedding"] for item in response.json()["data"]]

## Load the dataset

We'll use the [StockImages-CC0](https://huggingface.co/datasets/KoalaAI/StockImages-CC0) dataset, a collection of ~4,000 CC0-licensed stock photos with descriptive tags. Images are 1200px wide, well above CLIP v2's 512x512 input size.

We'll select a diverse subset of 20 images covering different categories.

In [ ]:
from datasets import load_dataset

full_dataset = load_dataset("KoalaAI/StockImages-CC0", split="train")
print(f"Total images: {len(full_dataset)}")

# Select 20 diverse images across different categories
selected_indices = [
    0,  # technology: smartphone, macbook
    8,  # coastal landscape: driftwood, sea, ocean
    34,  # waterfall: rock, waterfall, creek
    40,  # fashion: highheel, shoe, red
    61,  # vineyard: vine, wine, fruit
    82,  # fruit: raspberry, berry
    90,  # night sky: milky way, stars
    95,  # music: acoustic guitar
    111,  # town: hot air balloon
    120,  # vehicle: vw van, vintage
    150,  # city: eiffel tower, paris
    153,  # animal: puppy, canine
    191,  # sport: skateboard, kickflip
    197,  # drink: tea, honey
    286,  # wildlife: brown bear
    305,  # architecture: palace, cathedral
    312,  # coffee: latte, cappuccino
    317,  # flowers: tulip, bouquet
    371,  # nature: waterfall, river, cascade
    418,  # pet: kitten, cat
]

dataset = full_dataset.select(selected_indices)
print(f"Selected {len(dataset)} images")
dataset

In [ ]:
# Preview some images
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, ax in enumerate(axes):
    ax.imshow(dataset[i]["image"])
    ax.set_title(dataset[i]["tags"][:40], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Generate image embeddings

In [ ]:
images = [item["image"].convert("RGB") for item in dataset]

In [ ]:
# Encode all images in a single API call
image_embeddings = encode_images(images)
print(
    f"Generated {len(image_embeddings)} embeddings of {len(image_embeddings[0])} dimensions"
)

## Index documents

In [ ]:
from elasticsearch import helpers


def build_bulk_actions(dataset, image_embeddings, index_name):
    for i, item in enumerate(dataset):
        yield {
            "_index": index_name,
            "_id": i,
            "_source": {
                "image_embedding": image_embeddings[i],
                "tags": item.get("tags", ""),
            },
        }


success, failed = helpers.bulk(
    es_client,
    build_bulk_actions(dataset, image_embeddings, INDEX_NAME),
    refresh=True,
)

print(f"Indexed {success} documents")

## Multilingual text-to-image search

CLIP v2's multilingual support means we can search with the **same query in different languages** and get consistent results.

In [ ]:
def search_by_text(query, k=5):
    """Encode a text query and search Elasticsearch."""

    query_embedding = encode_texts([query])[0]

    results = es_client.search(
        index=INDEX_NAME,
        knn={
            "field": "image_embedding",
            "query_vector": query_embedding,
            "k": k,
            "num_candidates": 50,
        },
    )

    return results["hits"]["hits"]


def display_results(hits, query=""):
    """Display search results as images."""

    n = len(hits)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))

    if n == 1:
        axes = [axes]
    if query:
        fig.suptitle(f'Query: "{query}"', fontsize=14, y=1)

    for _, (ax, hit) in enumerate(zip(axes, hits)):
        idx = int(hit["_id"])
        ax.imshow(dataset[idx]["image"])
        ax.set_title(
            f'Score: {hit["_score"]:.3f}',
            fontsize=10,
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

### Same query, different languages

In [ ]:
multilingual_queries = [
    {
        "English": "a cat sleeping",
        "Spanish": "un gato durmiendo",
        "French": "un chat qui dort",
        "Portuguese": "um gato dormindo",
    },
    {
        "English": "red flowers",
        "Spanish": "flores rojas",
        "French": "fleurs rouges",
        "Portuguese": "flores vermelhas",
    },
    {
        "English": "waterfall in nature",
        "Spanish": "cascada en la naturaleza",
        "French": "cascade dans la nature",
        "Portuguese": "cascata na natureza",
    },
]

for query_set in multilingual_queries:
    print(f"\n{'='*60}")

    for lang, query in query_set.items():
        print(f'\n{lang}: "{query}"')
        hits = search_by_text(query, k=3)
        display_results(hits, query=f"[{lang}] {query}")

## Image-to-image search

We can also use an image as the query to find visually similar products.

In [ ]:
def search_by_image(image, k=5):
    """Encode an image and search Elasticsearch."""

    query_embedding = encode_images([image])[0]

    results = es_client.search(
        index=INDEX_NAME,
        knn={
            "field": "image_embedding",
            "query_vector": query_embedding,
            "k": k,
            "num_candidates": 50,
        },
    )

    return results["hits"]["hits"]

In [ ]:
# Use a sample image as query
query_image = dataset[10]["image"]

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(query_image)
ax.set_title("Query image")
ax.axis("off")
plt.show()

hits = search_by_image(query_image)
display_results(hits, query="Similar to query image")

## Matryoshka Representations

Jina CLIP v2 supports [Matryoshka Representation Learning](https://arxiv.org/abs/2205.13147). Embeddings can be **truncated to smaller dimensions** while preserving most of their quality. This is useful for reducing storage and speeding up search.

In [ ]:
# The Jina API supports Matryoshka natively via the `dimensions` parameter
query = "black sneakers"

for dim in [1024, 512, 256, 128]:
    embedding = encode_texts([query], dimensions=dim)[0]
    print(f"Dimensions: {dim} | Embedding size: {len(embedding)}")

### Experimenting with reduced dimensions

Let's create a second index with 256 dimensions and compare search results against the full 1024-dimension index.

In [ ]:
MATRYOSHKA_DIMS = 256
MATRYOSHKA_INDEX = "clip-v2-stock-images-256d"

if es_client.indices.exists(index=MATRYOSHKA_INDEX):
    es_client.indices.delete(index=MATRYOSHKA_INDEX)

es_client.indices.create(
    index=MATRYOSHKA_INDEX,
    mappings={
        "properties": {
            "image_embedding": {
                "type": "dense_vector",
                "dims": MATRYOSHKA_DIMS,
                "index": True,
                "similarity": "cosine",
            },
            "tags": {
                "type": "text",
                "fields": {"keyword": {"type": "keyword"}},
            },
        }
    },
)

# Generate 256-dim embeddings
image_embeddings_256 = encode_images(images, dimensions=MATRYOSHKA_DIMS)
print(
    f"Generated {len(image_embeddings_256)} embeddings of {len(image_embeddings_256[0])} dimensions"
)

# Index documents
success, _ = helpers.bulk(
    es_client,
    build_bulk_actions(dataset, image_embeddings_256, MATRYOSHKA_INDEX),
    refresh=True,
)
print(f"Indexed {success} documents in {MATRYOSHKA_INDEX}")

In [ ]:
# Compare results: 1024 dims vs 256 dims
query = "a cat sleeping"

print("Results with 1024 dimensions:")
hits_1024 = search_by_text(query, k=3)
display_results(hits_1024, query=f"{query} (1024 dims)")

print("\nResults with 256 dimensions:")
query_embedding_256 = encode_texts([query], dimensions=MATRYOSHKA_DIMS)[0]
hits_256 = es_client.search(
    index=MATRYOSHKA_INDEX,
    knn={
        "field": "image_embedding",
        "query_vector": query_embedding_256,
        "k": 3,
        "num_candidates": 50,
    },
)["hits"]["hits"]
display_results(hits_256, query=f"{query} (256 dims)")

# Compare ranking
ids_1024 = [hit["_id"] for hit in hits_1024]
ids_256 = [hit["_id"] for hit in hits_256]
print(f"1024d ranking: {ids_1024}")
print(f" 256d ranking: {ids_256}")
print(f"Same top results: {ids_1024 == ids_256}")

## Cleanup

In [ ]:
es_client.indices.delete(index=INDEX_NAME)
es_client.indices.delete(index=MATRYOSHKA_INDEX, ignore=[404])
print(f"Deleted indices: {INDEX_NAME}, {MATRYOSHKA_INDEX}")